# 01 – Datenaufbereitung

Ziel: Die BFS-Rohdaten einlesen, in eine saubere Tabelle bringen und auf ein gemeinsames Basisjahr umrechnen, damit sich Mieten, Löhne und Teuerung vergleichen lassen.

Zwei Quellen:
- **Mietpreisindex** (`je-d-05.06.01.01.xlsx`) – verschachteltes Excel-Layout, muss umgeformt werden.
- **Löhne + Konsumentenpreise** (`ts-x-03.04.03.02.01.csv`) – bereits sauberes Langformat.

In [ ]:
import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"

## Teil A – Mietpreisindex

### A1. Rohdaten ansehen

Wir lesen ganz ohne Header ein (`header=None`), weil die Datei Titelzeilen, Leerzeilen und mehrere Jahresblöcke nebeneinander enthält.

In [ ]:
raw = pd.read_excel(f"{RAW}/je-d-05.06.01.01.xlsx", header=None)
print("Form:", raw.shape)
raw.head(20)

### A2. Das Problem

Die Jahre stehen **horizontal in Blöcken** (1984–1994, dann 1995–2005, ...). Pro Block gibt es eine Zeile mit den Jahren und – drei Zeilen darunter – die Zeile `Mietpreisindex (déc. 2015=100)` mit den Werten.

**Vorgehen:** Wir durchlaufen alle Zeilen, erkennen die Jahres-Zeilen (Spalte 0 leer, Spalte 1 eine Jahreszahl) und lesen die zugehörigen Indexwerte aus der Zeile 3 weiter unten. So entsteht eine saubere Tabelle `Jahr | Mietindex`.

In [ ]:
records = []
for i, row in raw.iterrows():
    ist_jahreszeile = (
        pd.isna(row[0])
        and pd.notna(row[1])
        and str(row[1]).replace(".0", "").isdigit()
        and int(float(row[1])) > 1900
    )
    if ist_jahreszeile:
        jahre = row[1:].tolist()
        index_werte = raw.iloc[i + 3, 1:].tolist()  # Indexzeile liegt 3 Zeilen tiefer
        for jahr, wert in zip(jahre, index_werte):
            if pd.notna(jahr) and pd.notna(wert):
                records.append((int(float(jahr)), float(wert)))

miete = pd.DataFrame(records, columns=["Jahr", "Mietindex"]).sort_values("Jahr").reset_index(drop=True)
print(f"{len(miete)} Jahre: {miete['Jahr'].min()}–{miete['Jahr'].max()}")
miete.head()

## Teil B – Löhne und Konsumentenpreise

Diese Datei ist bereits sauber. Spalten: `YEAR`, `SEX` (T=Total, M=Männer, W=Frauen), `WAGE_TYPE` (N=Nominallohn, R=Reallohn), `VALUE` (Indexwert), `CPI` (jährliche Teuerung in %).

Wir nehmen nur **Total** und ziehen Nominal- und Reallohn nebeneinander.

In [ ]:
df = pd.read_csv(f"{RAW}/ts-x-03.04.03.02.01.csv", sep=";")
total = df[df["SEX"] == "T"]

nominal = total[total["WAGE_TYPE"] == "N"][["YEAR", "VALUE"]].rename(columns={"YEAR": "Jahr", "VALUE": "Nominallohn"})
real = total[total["WAGE_TYPE"] == "R"][["YEAR", "VALUE"]].rename(columns={"YEAR": "Jahr", "VALUE": "Reallohn"})

loehne = nominal.merge(real, on="Jahr")
print(f"Löhne: {loehne['Jahr'].min()}–{loehne['Jahr'].max()}")
loehne.tail()

## Teil C – Zusammenführen und vergleichbar machen

Miete und Löhne haben **unterschiedliche Basisjahre** (Miete: Dez. 2015=100; Löhne: historische Basis). Damit man die Kurven vergleichen kann, rechnen wir alle auf ein **gemeinsames Basisjahr 2010=100** um.

Wir mergen über das Jahr (`inner`, also nur Jahre, die in beiden vorkommen).

In [ ]:
vergleich = miete.merge(loehne, on="Jahr", how="inner")

BASISJAHR = 2010
for spalte in ["Mietindex", "Nominallohn", "Reallohn"]:
    basiswert = vergleich.loc[vergleich["Jahr"] == BASISJAHR, spalte].values[0]
    vergleich[f"{spalte}_idx"] = (vergleich[spalte] / basiswert * 100).round(1)

vergleich[["Jahr", "Mietindex_idx", "Nominallohn_idx", "Reallohn_idx"]].tail(10)

## Teil D – Bereinigten Datensatz speichern

Diese Datei nutzen wir in `02_analyse.ipynb` und später als Datenquelle für Power BI.

In [ ]:
vergleich.to_csv(f"{PROCESSED}/miete_loehne_vergleich.csv", index=False)
print("Gespeichert:", f"{PROCESSED}/miete_loehne_vergleich.csv")